# Beer Trendline & Popularity Prediction (1996-2018)**Group 14 -- Big Data Tools & Management (Waterloo Data Science Certificate)**Team: Mohammad Ali Bhatti, Arun Arora, Darwin Arriagada, Supratim Das, Mary (Zhong) Xu## ObjectiveUsing a public Kaggle dataset of beers, breweries, and ~9M user reviews spanning 1996-2018, the goalwas to clean and join three linked datasets, engineer a unified popularity score for each beer, andtrain a regression model to predict that score from beer/brewery attributes.Dataset source: [Brewery Dataset on Kaggle](https://www.kaggle.com/datasets/ankurnapa/brewery-dataset)(`beers.csv`, `breweries.csv`, `reviews.csv` -- not included in this repo due to size; see README fordownload instructions).

## Setup

In [ ]:
import numpy as npimport pandas as pd

## Data Loading & Exploration

### Beers

In [ ]:
beer_df = pd.read_csv('beers.csv')beer_df.shape

Sample the contents of the file.

In [ ]:
beer_df.head(20)

In [ ]:
beer_df.info()

How many beers of each style are in the dataset?

In [ ]:
beer_style_cnt = beer_df.groupby(['style'])['style'].count().sort_values(ascending=False)print(beer_style_cnt.to_string())

The most prolific style is American IPA, followed by American Pale Ale. The rarest styles are Wild/Sour Beers. This roughly tracks what we'd expect in the real world, with Ale-style beers being the most abundant and easiest to find.

How many distinct beer styles are listed?

In [ ]:
beer_df['style'].nunique()

112 distinct styles -- a wide variety, which means high dimensionality if we encode style directly as a feature.

### Breweries

In [ ]:
breweries_df = pd.read_csv('breweries.csv')breweries_df.shape

Sample the contents of the file.

In [ ]:
breweries_df.head(20)

In [ ]:
breweries_df.info()

How many brewery types are there per country and state?

In [ ]:
breweries_by_country_df = breweries_df.groupby(['country', 'state', 'types'])['id'].count()print(breweries_by_country_df.to_string())

There's one odd row: country `HR` with state `GB2`, which looks like a data entry error since `GB2` is otherwise a UK state code.

In [ ]:
breweries_df[breweries_df['country'] == 'HR']

From the city name, `HR` is actually Croatia -- the state field for this row is just mislabeled, not a true country mismatch.

In [ ]:
breweries_df[breweries_df['country'] == 'GB']

### Reviews

In [ ]:
reviews_df = pd.read_csv('reviews.csv')reviews_df.shape

In [ ]:
reviews_df.head(20)

In [ ]:
reviews_df.info()

What's the time span of the reviews, and how many came in per date?

In [ ]:
reviews_by_date_df = reviews_df.groupby(['date'])['date'].count()print(reviews_by_date_df.to_string())

The oldest review is from 22 Aug 1996; the most recent is 30 Sep 2018 -- about 22 years of data. Volume is sparse in the early years, picks up past ~100/day from late 2002, and crosses 1,000/day by mid-2011.

## Joining the Datasets

Rename `id` in **beers** to `beer_id` so it matches the join key in **reviews**.

In [ ]:
beer_df.rename(columns={'id': 'beer_id'}, inplace=True)beer_df.head()

Check whether every beer has at least one review.

In [ ]:
beers_reviewed_df = beer_df.merge(reviews_df.drop_duplicates(), on=['beer_id'], how='left', indicator=True)no_review_df = beers_reviewed_df[beers_reviewed_df['_merge'] == 'left_only']no_review_df['beer_id'].unique().shape

Of 358,873 beers listed, 49,348 have no rating at all. These carry none of the signal we care about and get dropped before training.

Rename `id` in **breweries** to `brewery_id` to match **beers**.

In [ ]:
breweries_df.rename(columns={'id': 'brewery_id'}, inplace=True)breweries_df.head()

Join **beers** to **breweries** and check for orphan beers with no matching brewery.

In [ ]:
breweries_beer_df = beer_df.merge(breweries_df, on=['brewery_id'], how='left', indicator=True)beer_wout_brewery_df = breweries_beer_df[breweries_beer_df['_merge'] == 'left_only']beer_wout_brewery_df['brewery_id'].unique().shape

No orphans -- every beer resolves to a brewery, so location data is available across the board.

#### Sanity check: does state/country agree between the beer and brewery records?

In [ ]:
conditions = [    (breweries_beer_df['state_x'] != breweries_beer_df['state_y']),    (breweries_beer_df['country_x'] != breweries_beer_df['country_y']),    (breweries_beer_df['state_x'] == breweries_beer_df['state_y']),    (breweries_beer_df['country_x'] == breweries_beer_df['country_y']),]mismatch_flags = ['state', 'cntry', 'same_state', 'same_cntry']breweries_beer_df['cnty_ste_match'] = np.select(conditions, mismatch_flags, default='unknown')breweries_beer_df.head(50)

In [ ]:
cntry_ste_mismatch_df = breweries_beer_df.loc[    (breweries_beer_df['cnty_ste_match'] == 'state') | (breweries_beer_df['cnty_ste_match'] == 'cntry')]cntry_ste_mismatch_df.groupby(['cnty_ste_match']).size()

No true country mismatches. The flagged rows are all cases where `state` is `NaN` (60,726 rows) rather than an actual conflict.

#### Now combine all three datasets

In [ ]:
reviews_beer_df = pd.merge(reviews_df, beer_df, how='left', on='beer_id')reviews_beer_brewery_df = pd.merge(reviews_beer_df, breweries_df, how='left', on='brewery_id')reviews_beer_brewery_df.columns

### Engineer the target variable: `custom_score`

The dataset's own `score` field doesn't document how it's calculated, so we build our own average across the five rated attributes (look, smell, taste, feel, overall) and use that as the prediction target instead.

In [ ]:
reviews_beer_brewery_df['custom_score'] = (    (reviews_beer_brewery_df['look']     + reviews_beer_brewery_df['smell']     + reviews_beer_brewery_df['taste']     + reviews_beer_brewery_df['feel']     + reviews_beer_brewery_df['overall']) / 5).round(2)reviews_beer_brewery_df.head()

A handful of reviews are missing one or more of the five sub-ratings, which leaves `custom_score` as `NaN`. For those rows we fall back to the dataset's own `score`.

In [ ]:
reviews_beer_brewery_df['custom_score'] = reviews_beer_brewery_df['custom_score'].fillna(    reviews_beer_brewery_df.pop('score'))reviews_beer_brewery_df['custom_score'].isna().sum()

Drop the columns that won't be used for modeling -- free-text fields, the now-redundant rating sub-scores, and duplicate/low-value metadata from the joins.

In [ ]:
reviews_beer_brewery_df.drop(    columns=['username', 'text', 'look', 'smell', 'taste', 'feel', 'overall',             'availability', 'notes_x', 'retired', 'name_y', 'notes_y', 'types'],    inplace=True)reviews_beer_brewery_df.head()

### Encode beer style as a numeric feature

In [ ]:
unique_styles = sorted(reviews_beer_brewery_df['style'].dropna().unique())print('There are', len(unique_styles), 'unique styles of beer')

In [ ]:
style_numerical = dict(zip(range(len(unique_styles)), unique_styles))style_numerical

Quick sanity check on the lookup table -- e.g. what style does key `11` map to?

In [ ]:
style_numerical[11]

In [ ]:
style_numerical_reversed = {v: k for k, v in style_numerical.items()}reviews_beer_brewery_df['numerical_style'] = reviews_beer_brewery_df['style'].map(style_numerical_reversed)reviews_beer_brewery_df.head()

## Model Training & EvaluationWith the joined, feature-engineered dataset in place, the team moved into Databricks to train andevaluate regression models predicting `custom_score` from beer/brewery attributes, using Spark MLlib.The Databricks cluster available to the team couldn't handle a full-size split, so training was cappedat a random 0.5% sample, with a standard 70/30 train/test split.

In [ ]:
# modeling step reconstructed from the final report -- original Databricks run wasn't exportedfrom pyspark.sql import SparkSessionfrom pyspark.ml.feature import StringIndexer, VectorAssemblerfrom pyspark.ml.regression import LinearRegression, RandomForestRegressorfrom pyspark.ml.evaluation import RegressionEvaluatorspark = SparkSession.builder.appName('G14-BeerPopularity').getOrCreate()# bring the engineered pandas dataframe into Sparkspark_df = spark.createDataFrame(reviews_beer_brewery_df)# the team's Databricks cluster couldn't handle a full split, so training was# capped at a 0.5% random sample, matching what's documented in the final reportsample_df = spark_df.sample(withReplacement=False, fraction=0.005, seed=42)categorical_cols = ['style', 'country_x', 'state_x']indexers = [    StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep')    for c in categorical_cols]assembler = VectorAssembler(    inputCols=['abv'] + [f'{c}_idx' for c in categorical_cols],    outputCol='features',    handleInvalid='skip')pipeline_df = sample_dffor indexer in indexers:    pipeline_df = indexer.fit(pipeline_df).transform(pipeline_df)pipeline_df = assembler.transform(pipeline_df)train_df, test_df = pipeline_df.randomSplit([0.7, 0.3], seed=42)

### Linear Regression

In [ ]:
lr = LinearRegression(featuresCol='features', labelCol='custom_score')lr_model = lr.fit(train_df)lr_predictions = lr_model.transform(test_df)lr_evaluator = RegressionEvaluator(labelCol='custom_score', predictionCol='prediction', metricName='r2')lr_r2 = lr_evaluator.evaluate(lr_predictions)lr_rmse = lr_evaluator.setMetricName('rmse').evaluate(lr_predictions)print(f'Linear Regression -- R2: {lr_r2:.2f}, RMSE: {lr_rmse:.2f}')

### Random Forest

In [ ]:
rf = RandomForestRegressor(featuresCol='features', labelCol='custom_score', seed=42)rf_model = rf.fit(train_df)rf_predictions = rf_model.transform(test_df)rf_evaluator = RegressionEvaluator(labelCol='custom_score', predictionCol='prediction', metricName='r2')rf_r2 = rf_evaluator.evaluate(rf_predictions)rf_rmse = rf_evaluator.setMetricName('rmse').evaluate(rf_predictions)print(f'Random Forest -- R2: {rf_r2:.2f}, RMSE: {rf_rmse:.2f}')

## Results & ConclusionPer the team's final report, Linear Regression and Random Forest were compared on the held-out testset using MSE, R2, and explained variance. Additional hyperparameter tuning on the Random Forest modelproduced only marginal gains, and **Linear Regression came out ahead with R2 = 0.70**.> "The group tested the performance of holdouts with Linear Regression, and with additional parameter> hypertuning in Random Forest Classifier by comparing MSE, R2, and explained variance. Little> improvements were seen in the model with increased hypertuning, and Linear Regression performed> marginally better with a greater R2 of 70%."> -- Group 14 Report Final**Limitations & future work (from the report):** combining three datasets introduced more variablesthan the team anticipated, and picking one more contained dataset up front might have simplifiedpreprocessing. Given more time, the team wanted to try Gradient Boosted models and explore platformsbetter suited to large-scale Spark ML workflows than what was available for this project.